# **Libreries**

In [ ]:
!pip uninstall -y transformers peft accelerate
!pip install -U peft accelerate
!pip install -U git+https://github.com/huggingface/transformers.git
!pip install -U qwen-vl-utils

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 127.4 MB/s eta 0:00:00
  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-m7f64wpi
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-m7f64wpi
  Resolved https://github.com/huggingface/transformers.git to commit d63bb4ac4a24b2f75244cf586919b18223506a4e
  Installing build dependencies ... done
  Getting requirements to buil

# **Load Model**

In [ ]:
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

model_id = "Qwen/Qwen2.5-VL-3B-Instruct"

processor = AutoProcessor.from_pretrained(model_id)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

# **VLM Function**

In [ ]:
def build_vr_prompt(question: str) -> str:
    return f"""
You are an AI assistant inside a VR chemistry laboratory.

The user selects an object highlighted in green at the center.

Your tasks:
1. Identify the selected object.
2. Answer the question in a short, practical way for a chemistry lab.
3. Decide if another object helps use the selected object.

Rules:
- If the question is only identification (e.g. "What is this?"):
  → related_object = null
  → relation_instruction = null
  → related_object_coordinates = null

- If the question is about usage:
  → include a related object ONLY if it helps

- Use realistic and safe lab logic:
  beaker/flask → liquids or chemicals
  burner → heating substances
  pipette → transferring liquids
  thermometer → measuring temperature
  scale → measuring mass

- Avoid unsafe or dangerous instructions
- Never invent unrealistic or harmful relations

Coordinates:
- Only for related object
- Must be tight bounding box
- If unsure → null
- Never full image

Return ONLY valid JSON.
No markdown.
No explanations.
Do not copy examples.

Example:
{{
  "object": "beaker",
  "answer": "You can use this beaker to hold or mix liquids.",
  "related_object": "burner",
  "relation_instruction": "Place the beaker on the burner to heat the liquid.",
  "related_object_coordinates": {{
    "x_min": 120,
    "y_min": 180,
    "x_max": 320,
    "y_max": 420
  }}
}}

User question:
{question}

JSON:
{{
  "object": "<object>",
  "answer": "<answer>",
  "related_object": "<object or null>",
  "relation_instruction": "<instruction or null>",
  "related_object_coordinates": {{
    "x_min": <int>,
    "y_min": <int>,
    "x_max": <int>,
    "y_max": <int>
  }} or null
}}
"""

In [ ]:
def build_vr_prompt(question: str) -> str:
    return f"""
You are an AI assistant inside a VR cooking game.

The player selects an object highlighted in green at the center.

Your tasks:
1. Identify the selected object.
2. Answer the question in a short, practical way for the cooking game.
3. Decide if another object helps use the selected object.

Rules:
- If the question is only identification (e.g. "What is this?"):
  → related_object = null
  → relation_instruction = null
  → related_object_coordinates = null

- If the question is about usage:
  → include a related object ONLY if it helps

- Use realistic kitchen logic:
  pot/pan/pressure cooker → stove
  knife → ingredients
  thermometer → food/liquid

- Never invent unrealistic or unsafe relations

Coordinates:
- Only for related object
- Must be tight bounding box
- If unsure → null
- Never full image

Return ONLY valid JSON.
No markdown.
No explanations.
Do not copy examples.

Example:
{{
  "object": "pot",
  "answer": "You can use this pot to cook food.",
  "related_object": "stove",
  "relation_instruction": "Place the pot on the stove to heat it.",
  "related_object_coordinates": {{
    "x_min": 100,
    "y_min": 200,
    "x_max": 300,
    "y_max": 400
  }}
}}

User question:
{question}

JSON:
{{
  "object": "<object>",
  "answer": "<answer>",
  "related_object": "<object or null>",
  "relation_instruction": "<instruction or null>",
  "related_object_coordinates": {{
    "x_min": <int>,
    "y_min": <int>,
    "x_max": <int>,
    "y_max": <int>
  }} or null
}}
"""

In [ ]:
def ask_vlm_single(image, question, max_new_tokens=220):
    prompt = build_vr_prompt(question)

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    )

    # Important for Qwen2.5-VL
    inputs = inputs.to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=processor.tokenizer.eos_token_id
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    print("DECODED OUTPUT:", repr(output_text))

    del inputs, generated_ids, generated_ids_trimmed
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return output_text

In [ ]:
import json
import re

def clean_model_json(raw_response):
    raw_response = raw_response.strip()
    raw_response = raw_response.replace("```json", "").replace("```", "").strip()

    match = re.search(r"\{.*\}", raw_response, re.DOTALL)
    if match:
        raw_response = match.group(0)

    try:
        return json.loads(raw_response)
    except:
        return {
            "object": "unknown",
            "answer": raw_response,
            "related_object": None,
            "relation_instruction": None,
            "related_object_coordinates": None
        }

# **API**

In [ ]:
# FastAPI
from fastapi import FastAPI, UploadFile, Form
from fastapi.middleware.cors import CORSMiddleware

# Imagen
from PIL import Image
from io import BytesIO

# Utilidades
import json


app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/")
def home():
    return {"message": "VLM running"}


import json

@app.post("/ask")
async def ask_model(
    image: UploadFile,
    question: str = Form(...)
):
    print("QUESTION RECEIVED:", question)

    image_bytes = await image.read()
    print("IMAGE BYTES:", len(image_bytes))

    pil_image = Image.open(BytesIO(image_bytes)).convert("RGB")
    pil_image.thumbnail((512, 512))

    raw_response = ask_vlm_single(pil_image, question)

    print("RAW MODEL RESPONSE START")
    print(repr(raw_response))
    print("RAW MODEL RESPONSE END")

    parsed_response = clean_model_json(raw_response)

    print("PARSED RESPONSE:")
    print(parsed_response)

    return parsed_response

# **Server**

In [ ]:
!pip install pyngrok

In [ ]:
import nest_asyncio
import uvicorn
from pyngrok import ngrok
import threading
import gc

nest_asyncio.apply()

# 🔐 PON TU TOKEN AQUÍ
ngrok.set_auth_token("3CwfczyUZrjbC99EZObapzccpav_2aUmVv661D8LipHYa5M8Y")

# Cerrar túneles anteriores si existen
ngrok.kill()

public_url = ngrok.connect(8000)

print("🚀 SERVER READY")
print("👉 Unity endpoint:", public_url.public_url + "/ask")

# Ejecutar servidor en background
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server)
server_thread.start()

🚀 SERVER READY
👉 Unity endpoint: https://unbundle-gothic-perjurer.ngrok-free.dev/ask


# **Test**

In [ ]:
import requests
import json

url = "https://unbundle-gothic-perjurer.ngrok-free.dev/ask"

files = {
    "image": open("/content/VLLMTEST.png", "rb")
}

data = {
    "question": "Tell me which object in this scene can I cut with this knife?"
}

response = requests.post(url, files=files, data=data)

# Convertir a JSON
response_json = response.json()

# Imprimir bonito
print(json.dumps(response_json, indent=4, ensure_ascii=False))

QUESTION RECEIVED: Tell me which object in this scene can I cut with this knife?
IMAGE BYTES: 218970
DECODED OUTPUT: '```json\n{\n  "object": "knife",\n  "answer": "You can use this knife to cut the tomato.",\n  "related_object": "tomato",\n  "relation_instruction": "Use the knife to cut the tomato.",\n  "related_object_coordinates": {\n    "x_min": 50,\n    "y_min": 150,\n    "x_max": 70,\n    "y_max": 170\n  }\n}\n```'
RAW MODEL RESPONSE START
'```json\n{\n  "object": "knife",\n  "answer": "You can use this knife to cut the tomato.",\n  "related_object": "tomato",\n  "relation_instruction": "Use the knife to cut the tomato.",\n  "related_object_coordinates": {\n    "x_min": 50,\n    "y_min": 150,\n    "x_max": 70,\n    "y_max": 170\n  }\n}\n```'
RAW MODEL RESPONSE END
PARSED RESPONSE:
{'object': 'knife', 'answer': 'You can use this knife to cut the tomato.', 'related_object': 'tomato', 'relation_instruction': 'Use the knife to cut the tomato.', 'related_object_coordinates': {'x_min':

In [ ]:
import requests

url = "https://unbundle-gothic-perjurer.ngrok-free.dev/ask"

files = {
    "image": open("/content/pan.png", "rb")
}

data = {
    "question": "What is this?"
}

response = requests.post(url, files=files, data=data)

print(response.json())

In [ ]:
import requests

url = "https://unbundle-gothic-perjurer.ngrok-free.dev/ask"

files = {
    "image": open("/content/pan.png", "rb")
}

data = {
    "question": "What can I use this object for in the cooking game?"
}

response = requests.post(url, files=files, data=data)

print(response.json())